# XGBoost Revenue Prediction

**Predict monthly revenue for each site using XGBoost**

This notebook provides a streamlined approach to:
- Load pre-processed site training data
- Train an XGBoost regressor to predict `avg_monthly_revenue`
- Evaluate model performance
- Generate predictions for all sites

## Why XGBoost?
- Handles mixed feature types (numeric, categorical) natively
- Built-in handling of missing values
- Often outperforms neural networks on tabular data with <100K rows
- Fast training and inference
- Interpretable feature importance

In [1]:
import polars as pl
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import json
import pickle
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')

print(f"XGBoost version: {xgb.__version__}")
print(f"Polars version: {pl.__version__}")

XGBoostError: 
XGBoost Library (libxgboost.dylib) could not be loaded.
Likely causes:
  * OpenMP runtime is not installed
    - vcomp140.dll or libgomp-1.dll for Windows
    - libomp.dylib for Mac OSX
    - libgomp.so for Linux and other UNIX-like OSes
    Mac OSX users: Run `brew install libomp` to install OpenMP runtime.

  * You are running 32-bit Python on a 64-bit OS

Error message(s): ["dlopen(/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/lib/libxgboost.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib\n  Referenced from: <58FE87DD-A5B4-3D80-BC4B-11FC831B9707> /Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/xgboost/lib/libxgboost.dylib\n  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/O

## 1. Load Data

In [ ]:
# Load pre-processed training data
DATA_PATH = 'site_training_data.parquet'
OUTPUT_DIR = '/outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

df = pl.read_parquet(DATA_PATH)
print(f"Loaded {len(df):,} sites with {len(df.columns)} columns")

# Target variable
TARGET = 'avg_monthly_revenue'
print(f"\nTarget: {TARGET}")
print(f"  Mean: ${df[TARGET].mean():,.2f}")
print(f"  Median: ${df[TARGET].median():,.2f}")

## 2. Define Features

We'll use three types of features:
- **Numeric**: Revenue metrics, demographics, distances
- **Categorical**: Network, program, hardware type, brands
- **Boolean**: Capability and restriction flags

In [ ]:
# Define feature groups
NUMERIC_FEATURES = [
    # Relative strength indicators
    'rs_Impressions', 'rs_NVIs', 'rs_Revenue', 'rs_RevenuePerScreen',
    # Revenue context
    'log_total_revenue',
    # Geospatial
    'log_nearest_site_distance_mi', 'log_min_distance_to_interstate_mi',
    'log_min_distance_to_kroger_mi', 'log_min_distance_to_mcdonalds_mi',
    # Demographics
    'avg_household_income', 'median_age', 'pct_female',
]

CATEGORICAL_FEATURES = [
    'network', 'program', 'experience_type', 'hardware_type',
    'retailer', 'brand_fuel', 'brand_restaurant', 'brand_c_store',
    'nearest_interstate',
]

# Boolean features (encoded as 0/1)
BOOLEAN_FEATURES = [col for col in df.columns if col.endswith('_encoded')]

print(f"Feature counts:")
print(f"  Numeric: {len(NUMERIC_FEATURES)}")
print(f"  Categorical: {len(CATEGORICAL_FEATURES)}")
print(f"  Boolean: {len(BOOLEAN_FEATURES)}")
print(f"  Total: {len(NUMERIC_FEATURES) + len(CATEGORICAL_FEATURES) + len(BOOLEAN_FEATURES)}")

In [ ]:
# Filter to available features
available_numeric = [f for f in NUMERIC_FEATURES if f in df.columns]
available_categorical = [f for f in CATEGORICAL_FEATURES if f in df.columns]
available_boolean = [f for f in BOOLEAN_FEATURES if f in df.columns]

print(f"Available features:")
print(f"  Numeric: {len(available_numeric)}")
print(f"  Categorical: {len(available_categorical)}")
print(f"  Boolean: {len(available_boolean)}")

# Check for missing numeric features
missing = set(NUMERIC_FEATURES) - set(available_numeric)
if missing:
    print(f"\n⚠️ Missing numeric features: {missing}")

## 3. Prepare Features for XGBoost

In [ ]:
# Convert to pandas for sklearn/xgboost compatibility
pdf = df.to_pandas()

# Encode categorical features
label_encoders = {}
for col in available_categorical:
    le = LabelEncoder()
    # Handle nulls by converting to string
    pdf[col] = pdf[col].fillna('__MISSING__').astype(str)
    pdf[f'{col}_encoded'] = le.fit_transform(pdf[col])
    label_encoders[col] = le

print(f"Encoded {len(label_encoders)} categorical features")

# Build feature matrix
feature_cols = (
    available_numeric + 
    [f'{col}_encoded' for col in available_categorical] + 
    available_boolean
)

# Remove target from features if present
feature_cols = [f for f in feature_cols if f != TARGET and f in pdf.columns]

print(f"\nTotal feature columns: {len(feature_cols)}")

In [ ]:
# Prepare X and y
X = pdf[feature_cols].copy()
y = pdf[TARGET].copy()

# Fill remaining nulls with median for numeric, 0 for boolean
for col in X.columns:
    if X[col].dtype in ['float64', 'float32']:
        X[col] = X[col].fillna(X[col].median())
    else:
        X[col] = X[col].fillna(0)

# Remove any rows with null target
valid_mask = ~y.isna()
X = X[valid_mask]
y = y[valid_mask]

print(f"Final dataset: {len(X):,} samples x {len(feature_cols)} features")
print(f"Target range: ${y.min():,.2f} - ${y.max():,.2f}")

## 4. Train/Validation/Test Split

In [ ]:
# Split: 70% train, 15% validation, 15% test
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.176, random_state=42  # 0.176 * 0.85 ≈ 0.15
)

print(f"Train: {len(X_train):,} samples ({len(X_train)/len(X)*100:.1f}%)")
print(f"Val:   {len(X_val):,} samples ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test:  {len(X_test):,} samples ({len(X_test)/len(X)*100:.1f}%)")

## 5. Train XGBoost Model

In [ ]:
# XGBoost parameters optimized for revenue prediction
params = {
    'objective': 'reg:squarederror',
    'eval_metric': 'mae',
    'max_depth': 8,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 5,
    'reg_alpha': 0.1,      # L1 regularization
    'reg_lambda': 1.0,     # L2 regularization
    'seed': 42,
    'n_jobs': -1,
}

# Create DMatrix for efficient training
dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_cols)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=feature_cols)
dtest = xgb.DMatrix(X_test, label=y_test, feature_names=feature_cols)

print("XGBoost Parameters:")
for k, v in params.items():
    print(f"  {k}: {v}")

In [ ]:
# Train with early stopping
print("Training XGBoost model...\n")

evals = [(dtrain, 'train'), (dval, 'val')]
evals_result = {}

model = xgb.train(
    params,
    dtrain,
    num_boost_round=1000,
    evals=evals,
    evals_result=evals_result,
    early_stopping_rounds=50,
    verbose_eval=100,
)

print(f"\n✓ Training complete!")
print(f"  Best iteration: {model.best_iteration}")
print(f"  Best validation MAE: ${model.best_score:.2f}")

In [ ]:
# Plot training curves
fig, ax = plt.subplots(figsize=(10, 5))

ax.plot(evals_result['train']['mae'], label='Train MAE', alpha=0.8)
ax.plot(evals_result['val']['mae'], label='Validation MAE', alpha=0.8)
ax.axvline(model.best_iteration, color='red', linestyle='--', label=f'Best: {model.best_iteration}')
ax.set_xlabel('Boosting Round')
ax.set_ylabel('MAE ($)')
ax.set_title('XGBoost Training Progress')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'xgboost_training_curve.png', dpi=150)
plt.show()

## 6. Evaluate Model Performance

In [ ]:
def evaluate_predictions(y_true, y_pred, dataset_name='Test'):
    """Calculate comprehensive regression metrics."""
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)
    
    # MAPE (handle zeros)
    mask = y_true > 0
    mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100
    
    # Median absolute error (robust to outliers)
    median_ae = np.median(np.abs(y_true - y_pred))
    
    print(f"{dataset_name} Set Metrics:")
    print("=" * 45)
    print(f"  MAE (Mean Absolute Error):     ${mae:,.2f}")
    print(f"  RMSE (Root Mean Square Error): ${rmse:,.2f}")
    print(f"  Median Absolute Error:         ${median_ae:,.2f}")
    print(f"  MAPE (Mean Abs % Error):       {mape:.1f}%")
    print(f"  R² Score:                      {r2:.4f}")
    
    return {'mae': mae, 'rmse': rmse, 'r2': r2, 'mape': mape, 'median_ae': median_ae}

# Generate predictions
y_train_pred = model.predict(dtrain)
y_val_pred = model.predict(dval)
y_test_pred = model.predict(dtest)

# Evaluate
train_metrics = evaluate_predictions(y_train.values, y_train_pred, 'Train')
print()
val_metrics = evaluate_predictions(y_val.values, y_val_pred, 'Validation')
print()
test_metrics = evaluate_predictions(y_test.values, y_test_pred, 'Test')

In [ ]:
# Visualize predictions
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs Predicted
ax = axes[0]
ax.scatter(y_test.values, y_test_pred, alpha=0.3, s=10)
max_val = max(y_test.values.max(), y_test_pred.max())
ax.plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='Perfect Prediction')
ax.set_xlabel('Actual Revenue ($)')
ax.set_ylabel('Predicted Revenue ($)')
ax.set_title(f'Actual vs Predicted (R² = {test_metrics["r2"]:.4f})')
ax.legend()
ax.grid(True, alpha=0.3)

# Residual distribution
ax = axes[1]
residuals = y_test.values - y_test_pred
ax.hist(residuals, bins=50, edgecolor='white', alpha=0.7)
ax.axvline(0, color='red', linestyle='--', linewidth=2)
ax.axvline(np.mean(residuals), color='green', linestyle='--', 
           label=f'Mean: ${np.mean(residuals):,.0f}')
ax.set_xlabel('Prediction Error ($)')
ax.set_ylabel('Count')
ax.set_title('Residual Distribution')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'xgboost_evaluation.png', dpi=150)
plt.show()

## 7. Feature Importance

In [ ]:
# Get feature importance (gain-based)
importance = model.get_score(importance_type='gain')

# Sort by importance
importance_df = pd.DataFrame([
    {'feature': k, 'importance': v} 
    for k, v in importance.items()
]).sort_values('importance', ascending=False)

print("Top 20 Most Important Features (by Gain):")
print("=" * 55)
for i, row in importance_df.head(20).iterrows():
    bar = '█' * int(row['importance'] / importance_df['importance'].max() * 20)
    print(f"{row['feature']:40} {row['importance']:8.1f} {bar}")

In [ ]:
# Plot feature importance
fig, ax = plt.subplots(figsize=(10, 8))

top_n = 20
top_features = importance_df.head(top_n)

ax.barh(range(len(top_features)), top_features['importance'].values[::-1])
ax.set_yticks(range(len(top_features)))
ax.set_yticklabels(top_features['feature'].values[::-1])
ax.set_xlabel('Importance (Gain)')
ax.set_title(f'Top {top_n} Feature Importances')
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'xgboost_feature_importance.png', dpi=150)
plt.show()

## 8. Generate Predictions for All Sites

In [ ]:
# Predict for all sites with valid targets (used in train/val/test splits)
dall = xgb.DMatrix(X, feature_names=feature_cols)
all_predictions = model.predict(dall)

# Create results dataframe with split labels
results_df = pdf[valid_mask][['id_gbase', 'gtvid', TARGET]].copy()
results_df['predicted_revenue'] = all_predictions
results_df['prediction_error'] = results_df[TARGET] - results_df['predicted_revenue']
results_df['abs_pct_error'] = (np.abs(results_df['prediction_error']) / 
                               (results_df[TARGET] + 1)) * 100

# Mark which split each site belongs to
results_df['split'] = 'unknown'
results_df.loc[X_train.index, 'split'] = 'train'
results_df.loc[X_val.index, 'split'] = 'val'
results_df.loc[X_test.index, 'split'] = 'test'

print(f"Generated predictions for {len(results_df):,} sites with known revenue")
print(f"\nSplit breakdown:")
print(f"  Train: {(results_df['split'] == 'train').sum():,}")
print(f"  Val:   {(results_df['split'] == 'val').sum():,}")
print(f"  Test:  {(results_df['split'] == 'test').sum():,}")

print(f"\nPrediction Accuracy (Test Set Only):")
test_results = results_df[results_df['split'] == 'test']
print(f"  Sites with <10% error: {(test_results['abs_pct_error'] < 10).sum():,} ({(test_results['abs_pct_error'] < 10).mean()*100:.1f}%)")
print(f"  Sites with <20% error: {(test_results['abs_pct_error'] < 20).sum():,} ({(test_results['abs_pct_error'] < 20).mean()*100:.1f}%)")
print(f"  Sites with <30% error: {(test_results['abs_pct_error'] < 30).sum():,} ({(test_results['abs_pct_error'] < 30).mean()*100:.1f}%)")

## 8b. Predict for Sites NOT in Training Data

The training parquet only contains ~26K active sites with sufficient history. Let's also generate predictions for ALL sites in the base dataset, including those that were excluded from training.

In [ ]:
# Load the full pre-cleaned dataset (includes inactive sites, sites with <12 months, etc.)
FULL_DATA_PATH = Path('../data/processed/site_aggregated_precleaned.parquet')

if FULL_DATA_PATH.exists():
    df_full = pl.read_parquet(FULL_DATA_PATH).to_pandas()
    print(f"Loaded full dataset: {len(df_full):,} total sites")
    
    # Find sites NOT in training data
    training_ids = set(pdf[valid_mask]['id_gbase'].tolist())
    df_new_sites = df_full[~df_full['id_gbase'].isin(training_ids)].copy()
    print(f"Sites NOT in training data: {len(df_new_sites):,}")
    
    if len(df_new_sites) > 0:
        # Prepare features for new sites (same encoding as training)
        for col in available_categorical:
            if col in df_new_sites.columns:
                df_new_sites[col] = df_new_sites[col].fillna('__MISSING__').astype(str)
                le = label_encoders[col]
                # Handle unseen categories
                df_new_sites[f'{col}_encoded'] = df_new_sites[col].apply(
                    lambda x: le.transform([x])[0] if x in le.classes_ else 0
                )
        
        # Build feature matrix for new sites
        X_new = pd.DataFrame()
        for col in feature_cols:
            if col in df_new_sites.columns:
                X_new[col] = df_new_sites[col]
            elif col.replace('_encoded', '') in available_categorical:
                # Already encoded above
                X_new[col] = df_new_sites[col]
            else:
                X_new[col] = 0  # Default for missing features
        
        # Fill nulls
        for col in X_new.columns:
            if X_new[col].dtype in ['float64', 'float32']:
                X_new[col] = X_new[col].fillna(X_new[col].median() if X_new[col].notna().any() else 0)
            else:
                X_new[col] = X_new[col].fillna(0)
        
        # Generate predictions
        dnew = xgb.DMatrix(X_new[feature_cols], feature_names=feature_cols)
        new_predictions = model.predict(dnew)
        
        # Create results for new sites
        new_results_df = df_new_sites[['id_gbase', 'gtvid', 'status']].copy()
        if 'avg_monthly_revenue' in df_new_sites.columns:
            new_results_df['actual_revenue'] = df_new_sites['avg_monthly_revenue']
        else:
            new_results_df['actual_revenue'] = np.nan
        new_results_df['predicted_revenue'] = new_predictions
        new_results_df['split'] = 'not_in_training'
        
        print(f"\n✓ Generated predictions for {len(new_results_df):,} new sites!")
        print(f"\nStatus breakdown of new sites:")
        print(new_results_df['status'].value_counts())
else:
    print(f"⚠️ Full dataset not found at {FULL_DATA_PATH}")
    print("Run data_transform.py first to generate the pre-cleaned dataset.")
    new_results_df = pd.DataFrame()

In [ ]:
# Display predictions for sites NOT in training data
if len(new_results_df) > 0:
    print("=" * 100)
    print("PREDICTIONS FOR SITES NOT IN TRAINING DATA")
    print("=" * 100)
    print(f"\nTotal sites: {len(new_results_df):,}")
    print(f"\nPredicted Revenue Distribution:")
    print(f"  Min:    ${new_results_df['predicted_revenue'].min():,.2f}")
    print(f"  Median: ${new_results_df['predicted_revenue'].median():,.2f}")
    print(f"  Mean:   ${new_results_df['predicted_revenue'].mean():,.2f}")
    print(f"  Max:    ${new_results_df['predicted_revenue'].max():,.2f}")
    
    # Show sample predictions with site IDs
    print(f"\n" + "-" * 100)
    print("Sample Predictions (Top 20 by Predicted Revenue):")
    print("-" * 100)
    print(f"{'GTVID':<15} {'ID_Gbase':<30} {'Status':<12} {'Predicted':>15}")
    print("-" * 100)
    
    top_new = new_results_df.nlargest(20, 'predicted_revenue')
    for _, row in top_new.iterrows():
        gtvid = str(row['gtvid'])[:14] if pd.notna(row['gtvid']) else 'N/A'
        id_gbase = str(row['id_gbase'])[:28] if pd.notna(row['id_gbase']) else 'N/A'
        status = str(row['status'])[:10] if pd.notna(row['status']) else 'N/A'
        pred = row['predicted_revenue']
        print(f"{gtvid:<15} {id_gbase:<30} {status:<12} ${pred:>13,.2f}")
    
    print(f"\n" + "-" * 100)
    print("Sample Predictions (Bottom 20 by Predicted Revenue):")
    print("-" * 100)
    
    bottom_new = new_results_df.nsmallest(20, 'predicted_revenue')
    for _, row in bottom_new.iterrows():
        gtvid = str(row['gtvid'])[:14] if pd.notna(row['gtvid']) else 'N/A'
        id_gbase = str(row['id_gbase'])[:28] if pd.notna(row['id_gbase']) else 'N/A'
        status = str(row['status'])[:10] if pd.notna(row['status']) else 'N/A'
        pred = row['predicted_revenue']
        print(f"{gtvid:<15} {id_gbase:<30} {status:<12} ${pred:>13,.2f}")

## 9. Save Model and Results

In [ ]:
# Save model
model.save_model(OUTPUT_DIR / 'xgboost_revenue_model.json')
print(f"✓ Model saved to: {OUTPUT_DIR / 'xgboost_revenue_model.json'}")

# Save label encoders
with open(OUTPUT_DIR / 'xgboost_label_encoders.pkl', 'wb') as f:
    pickle.dump(label_encoders, f)
print(f"✓ Label encoders saved to: {OUTPUT_DIR / 'xgboost_label_encoders.pkl'}")

# Combine ALL predictions (training sites + new sites)
if len(new_results_df) > 0:
    # Standardize column names for combining
    results_df_combined = results_df[['id_gbase', 'gtvid', TARGET, 'predicted_revenue', 'split']].copy()
    results_df_combined.rename(columns={TARGET: 'actual_revenue'}, inplace=True)
    
    new_results_combined = new_results_df[['id_gbase', 'gtvid', 'actual_revenue', 'predicted_revenue', 'split']].copy()
    
    # Combine
    all_predictions_df = pd.concat([results_df_combined, new_results_combined], ignore_index=True)
    print(f"\n✓ Combined predictions: {len(all_predictions_df):,} total sites")
    print(f"  - Training/Val/Test sites: {len(results_df):,}")
    print(f"  - New sites (not in training): {len(new_results_df):,}")
else:
    all_predictions_df = results_df[['id_gbase', 'gtvid', TARGET, 'predicted_revenue', 'split']].copy()
    all_predictions_df.rename(columns={TARGET: 'actual_revenue'}, inplace=True)

# Save ALL predictions
all_predictions_df.to_csv(OUTPUT_DIR / 'xgboost_all_predictions.csv', index=False)
print(f"✓ All predictions saved to: {OUTPUT_DIR / 'xgboost_all_predictions.csv'}")

# Also save just training predictions for backwards compatibility
results_df.to_csv(OUTPUT_DIR / 'xgboost_predictions.csv', index=False)
print(f"✓ Training predictions saved to: {OUTPUT_DIR / 'xgboost_predictions.csv'}")

# Save new site predictions separately
if len(new_results_df) > 0:
    new_results_df.to_csv(OUTPUT_DIR / 'xgboost_new_site_predictions.csv', index=False)
    print(f"✓ New site predictions saved to: {OUTPUT_DIR / 'xgboost_new_site_predictions.csv'}")

# Save feature importance
importance_df.to_csv(OUTPUT_DIR / 'xgboost_feature_importance.csv', index=False)
print(f"✓ Feature importance saved to: {OUTPUT_DIR / 'xgboost_feature_importance.csv'}")

# Save model summary
model_summary = {
    'model_type': 'XGBoost Regressor',
    'target': TARGET,
    'n_features': len(feature_cols),
    'n_training_samples': len(X),
    'n_new_site_predictions': len(new_results_df) if len(new_results_df) > 0 else 0,
    'n_total_predictions': len(all_predictions_df),
    'best_iteration': model.best_iteration,
    'params': params,
    'metrics': {
        'train': {k: float(v) for k, v in train_metrics.items()},
        'val': {k: float(v) for k, v in val_metrics.items()},
        'test': {k: float(v) for k, v in test_metrics.items()},
    },
    'feature_cols': feature_cols,
}

with open(OUTPUT_DIR / 'xgboost_model_summary.json', 'w') as f:
    json.dump(model_summary, f, indent=2)
print(f"✓ Model summary saved to: {OUTPUT_DIR / 'xgboost_model_summary.json'}")

## 10. Quick Inference Example

In [ ]:
def predict_revenue(model, site_data, feature_cols):
    """
    Predict revenue for new site(s).
    
    Args:
        model: Trained XGBoost model
        site_data: DataFrame with feature columns
        feature_cols: List of feature column names
    
    Returns:
        Array of predicted revenues
    """
    dmatrix = xgb.DMatrix(site_data[feature_cols], feature_names=feature_cols)
    return model.predict(dmatrix)

# Example: predict for first 5 sites
sample_indices = X.head(5).index
sample_sites = X.loc[sample_indices]
sample_predictions = predict_revenue(model, sample_sites, feature_cols)

# Get site IDs for display
sample_ids = pdf.loc[sample_indices, ['id_gbase', 'gtvid']]

print("Inference Example:")
print("=" * 90)
print(f"{'GTVID':<12} {'ID_Gbase':<28} {'Actual':>12} {'Predicted':>12} {'Error':>10}")
print("-" * 90)

for i, idx in enumerate(sample_indices):
    gtvid = sample_ids.loc[idx, 'gtvid']
    id_gbase = sample_ids.loc[idx, 'id_gbase']
    actual = y.loc[idx]
    pred = sample_predictions[i]
    error_pct = abs(actual - pred) / (actual + 1) * 100
    print(f"{gtvid:<12} {id_gbase:<28} ${actual:>10,.0f} ${pred:>10,.0f} {error_pct:>8.1f}%")

---

## Summary

This notebook has:
1. ✅ Loaded pre-processed site training data (~26K active sites)
2. ✅ Prepared numeric, categorical, and boolean features
3. ✅ Trained XGBoost regressor with early stopping
4. ✅ Evaluated model performance (MAE, RMSE, R², MAPE)
5. ✅ Analyzed feature importance
6. ✅ Generated predictions for training/val/test sites
7. ✅ **Generated predictions for ALL sites not in training data** (~30K+ additional sites)
8. ✅ Saved model and all artifacts

**Artifacts created:**
- `outputs/xgboost_revenue_model.json` - Trained model
- `outputs/xgboost_label_encoders.pkl` - Categorical encoders
- `outputs/xgboost_all_predictions.csv` - **ALL site predictions (training + new sites)**
- `outputs/xgboost_predictions.csv` - Training/val/test predictions only
- `outputs/xgboost_new_site_predictions.csv` - New sites not in training
- `outputs/xgboost_feature_importance.csv` - Feature importance
- `outputs/xgboost_model_summary.json` - Model configuration and metrics

**Prediction Output Format:**
| Column | Description |
|--------|-------------|
| `id_gbase` | MongoDB-style site identifier |
| `gtvid` | Business site identifier |
| `actual_revenue` | Known revenue (NaN for new sites) |
| `predicted_revenue` | Model prediction |
| `split` | `train`, `val`, `test`, or `not_in_training` |

**Next steps:**
- Compare with neural network model in `05_model_comparison.ipynb`
- Use SHAP for detailed feature explanations in `06_explainability.ipynb`